# 03_Feature_Engineering.ipynb
## FakeJobShield: Feature Extraction and Engineering Pipeline
This notebook extracts TF-IDF text features, encodes categorical metadata fields, extracts structured binary flags, and creates a combined sparse feature matrix for machine learning. It also exports the fitted TF-IDF Vectorizer and Label Encoders.


In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack, csr_matrix
import os

# Adjust working directory if run from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')


In [ ]:
# Load Cleaned Dataset
df = pd.read_csv("data/cleaned_fake_job_postings.csv")
df["cleaned_text"] = df["cleaned_text"].fillna("")
print("Loaded cleaned dataset shape:", df.shape)


In [ ]:
# Extract TF-IDF features
print("Fitting TF-IDF Vectorizer...")
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
x_tfidf = vectorizer.fit_transform(df["cleaned_text"])
print("TF-IDF matrix shape:", x_tfidf.shape)


In [ ]:
# Structured binary features
for col in ["telecommuting", "has_company_logo", "has_questions"]:
    df[col] = df[col].map({'t': 1, 'f': 0, '1': 1, '0': 0, 1: 1, 0: 0, True: 1, False: 0}).fillna(0).astype(int)

x_binary = df[["telecommuting", "has_company_logo", "has_questions"]].values
print("Binary features matrix shape:", x_binary.shape)


In [ ]:
# Categorical metadata features
cat_cols = ["employment_type", "required_experience", "required_education", "industry", "function"]
label_encoders = {}

encoded_cats = []
for col in cat_cols:
    df[col] = df[col].fillna("missing").astype(str)
    le = LabelEncoder()
    df[col + "_encoded"] = le.fit_transform(df[col])
    label_encoders[col] = le
    encoded_cats.append(df[col + "_encoded"].values.reshape(-1, 1))

x_categorical = np.hstack(encoded_cats)
print("Categorical features matrix shape:", x_categorical.shape)


In [ ]:
# Combine TF-IDF, binary, and categorical features
x_binary_sparse = csr_matrix(x_binary)
x_categorical_sparse = csr_matrix(x_categorical)

x_final = hstack([x_tfidf, x_binary_sparse, x_categorical_sparse])
print("Final Combined Feature Matrix shape:", x_final.shape)


In [ ]:
# Export the vectorizer, encoders, and pipelines
os.makedirs("models", exist_ok=True)
joblib.dump(vectorizer, "models/tfidf.pkl")
joblib.dump(label_encoders, "models/label_encoders.pkl")

feature_pipeline = {
    "vectorizer": vectorizer,
    "label_encoders": label_encoders,
    "cat_cols": cat_cols,
    "binary_cols": ["telecommuting", "has_company_logo", "has_questions"]
}
joblib.dump(feature_pipeline, "models/feature_pipeline.pkl")
print("Saved tfidf.pkl, label_encoders.pkl, and feature_pipeline.pkl")
